<a href="https://colab.research.google.com/github/Andres-Gress/Simulacion-I/blob/main/L%C3%ADnea_de_espera_con_un_servidor_Simu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Línea de Espera con un Servidor**

Programar el pseudocódigo propuesto por Sheldon Ross.

Como prueba, usar $\lambda = 4$ y Mu=6. Comentar y documentar.
Comparar las soluciones analítica y por simulación.


#### **Solución:**

---

##### **ANALÍTICA**

Para $\lambda = 4$ y $mu = 6$

Analíticamente se tienen las fórmulas:

$$\text{Factor de utilización}$$

$$\rho = \frac{\lambda}{\mu}$$

$$\text{Probabilidad de que no haya utilidades en el sistema}$$

$$P_0 = 1-\frac{\lambda}{\mu}$$

$$\text{Probabilidad de que haya n unidades en el sistema}$$

$$P_n = \frac{\lambda}{\mu}(P_0)$$

$$\text{Número promedio de unidades en cola}$$

$$L_q = \frac{\lambda^2}{\mu(\mu-\lambda)}$$

$$\text{Número promedio de unidades en el sistema}$$

$$L_s = L_q + \frac{\lambda}{\mu}$$

$$\text{Tiempo promedio que una uniad pasa en una cola}$$

$$W_q = \frac{L_q}{\lambda}$$

$$\text{Tiempo promedio en el sistema}$$

$$W = \frac{1}{\mu-\lambda}$$

Con los datos dados, analíticamente se tiene:

$\rho = \frac{2}{3}, \quad P_0 = \frac{1}{3}, \quad P_n =  \frac{2}{9}, \quad L_q = \frac{4}{3}, \quad L_s = 2, \quad W_q = \frac{1}{3}, \quad W = \frac{1}{2}$

##### **SIMULACIÓN**

Para simular el sistema anterior utilizamos las siguientes variables:



1.   Variable de tiempo $t$
2.   Variable de conteo $N_A: \text{ el número de llegadas hasta el instante } t$
$\\ \qquad \qquad \qquad \quad N_D: \text{ el número de salidas hasta el instante } t$
3.   Variable de estado del sistema $n: \text{ el número de clientes en el sistema }$

Hay 2 tipos de eventos: llegadas y salidas. Se tiene una lista de eventos la cual es: $$LE = t_A, t_D$$

Donde $ t_A $ es la hora de la siguiente llegada y $ t_D $ es la hora a la que concluye el servicio del cliente que se está atendiendo.

La variables de salida por registrar son $A(i)$, la hora de llegada del cliente $i$; $ D(i)$, la hora de salida del cliente $i$, y $T$, el tiempo de salida del último cliente, posterior a $T$.

Queremos:   

*   Tiempo promedio de un cliente en el sistema
*   Tiempo en que sale el último cliente

In [1]:
import numpy as np

In [2]:
def mm1(lam, mu, T):
    t = 0
    NA = 0 # número de llegadas
    ND = 0 # número de salidas
    n = 0 # clientes en el sistema
    tA = np.random.exponential(1/lam) # tiempo de llegada
    tD = np.inf # tiempo de salida

    A = []
    D = []

    nc_sistema = 0
    ne_cola = 0
    tm_llegadas = 0
    t_i = 0

    while True:
        # CASO 1
        if tA <= tD and tA <= T:
            t = tA
            dt = t - t_i
            nc_sistema += n * dt
            ne_cola += max(n - 1, 0) * dt
            if n > 0:
                tm_llegadas += dt
            t_i = t
            NA += 1
            n += 1
            A.append(t)
            tA = t + np.random.exponential(1/lam)
            if n == 1:
                Y = np.random.exponential(1/mu)
                tD = t + Y
        # CASO 2
        elif tD < tA and tD <= T:
            t = tD
            dt = t - t_i
            nc_sistema += n * dt
            ne_cola += max(n - 1, 0) * dt
            if n > 0:
                tm_llegadas += dt
            t_i = t
            n -= 1
            ND += 1
            D.append(t)
            if n == 0:
                tD = np.inf
            else:
                Y = np.random.exponential(1/mu)
                tD = t + Y
        # CASO 3
        elif min(tA, tD) > T and n > 0:
            t = tD
            dt = t - t_i
            if t_i < T:
                dt_real = min(t, T) - t_i
                nc_sistema += n * dt_real
                ne_cola += max(n - 1, 0) * dt_real
                if n > 0:
                    tm_llegadas += dt_real
            t_i = t
            n -= 1
            ND += 1
            D.append(t)
            if n > 0:
                Y = np.random.exponential(1/mu)
                tD = t + Y
        # CASO 4
        else:
            break

    W = np.array(D) - np.array(A)
    W_prom = np.mean(W)
    L_prom = nc_sistema / T
    Lq_prom = ne_cola / T
    rho = tm_llegadas / T
    W_q=rho*W_prom
    return {"Tiempo_promedio": W_prom, "L_prom": L_prom, "Lq_prom": Lq_prom, "rho": rho, "W_q": W_q, "Clientes_totales": NA, "Salidas_totales": ND}

Se crea una fución donde se aplican los casos que considera Sheldon Ross para esta línea de espera M/M/1.

Retorna los valores pedidos

In [4]:
lam = 4
mu = 6
T = 10000
s = mm1(lam, mu, T)

print("Eficiencia (rho) :", s["rho"])
print("Clientes promedio en sistema (L_s):", s["L_prom"])
print("Clientes promedio en cola (L_q):", s["Lq_prom"])
print("Tiempo promedio (W):", s["Tiempo_promedio"])
print("Tiempo promedio en cola (W_q):", s["W_q"])

Eficiencia (rho) : 0.6696787797978856
Clientes promedio en sistema (L_s): 2.0231594252444602
Clientes promedio en cola (L_q): 1.3534806454465642
Tiempo promedio (W): 0.5024735310064706
Tiempo promedio en cola (W_q): 0.3364958611251483


Nótese que coinciden los resultados por simulación con los analíticos, por lo que la función que simula la línea de espera es correcta y arroja resultados muy aproximados a la solución real.